## Particionando Dataset Eliteserien

In [1]:
import os
import shutil
import random
from pathlib import Path
import zipfile

In [3]:
with open("Eliteserien.zip", "rb") as f:
    print(f.read(100))

b'PK\x03\x04\x14\x00\x08\x08\x08\x00\x00\x00!\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00*\x00`\x00Eliteserien/2022/frames/6111/6111_0115.jpgUT\x05\x00\x01\xfcm\xa5e\n\x00 \x00\x00\x00\x00\x00\x01\x00\x18\x00\x00&\x07\xe4\xd9G\xda'


In [5]:
nome_zip = "Eliteserien.zip"

with zipfile.ZipFile(nome_zip, 'r') as zip_ref:
    zip_ref.extractall()

BadZipFile: File is not a zip file

O dataset possui 750 amostras balanceadas em 3 anos (ou seja, 250 amostras por ano)

Vamos selecionar aleatoriamente de modo que tenhamos 3 pastas no dataset particionado: treino (525), teste (131) e validação (95), de forma que fique 175 por ano (treinamento), 25 (validacao) e 50 (teste). Vamos contar pela pasta frames.

In [ ]:
caminho_dataset = Path("Eliteserien")
caminho_destino = Path("Eliteserien_Particionado")

In [ ]:
anos = ["2021", "2022", "2023"]
ids_especificos = []

for ano in caminho_dataset.iterdir():
  aux = []
  if ano.is_dir():
    caminho_pasta = Path(ano/"frames")
    for id_geral in caminho_pasta.iterdir():
      for id_especifico in id_geral.iterdir():
        aux.append(id_especifico.stem)
    ids_especificos.append(aux)

print(ids_especificos[0][0])


FileNotFoundError: [Errno 2] No such file or directory: 'Eliteserien'

In [ ]:
quantidade_treino = 175
quantidade_validacao = 25
quantidade_teste = 50

In [ ]:
categorias = {
    "detection": {"destino": "labels", "ext": ".txt"},
    "frames": {"destino": "frames", "ext": ".jpg"}
}

anos = ["2021", "2022", "2023"]

for ano in anos:
    ids_ano = []
    caminho_pasta = caminho_dataset / ano / "frames"
    
    if caminho_pasta.exists():
        for id_geral in caminho_pasta.iterdir():
            if id_geral.is_dir():
                for id_especifico in id_geral.iterdir():
                    ids_ano.append(id_especifico.stem)
            
    random.shuffle(ids_ano)

    splits = {
        "train": ids_ano[:quantidade_treino],
        "val": ids_ano[quantidade_treino:quantidade_treino+quantidade_validacao],
        "test": ids_ano[quantidade_treino+quantidade_validacao:]
    }

    for split_name, id_list in splits.items():
        for item_id in id_list:
            id_geral = item_id.split("_")[0]

            for cat_origem, info in categorias.items():
                src_file = caminho_dataset / ano / cat_origem / id_geral / f"{item_id}{info['ext']}"

                if src_file.exists():
                    # Alterado aqui: destino / categoria (frames ou labels) / split (train, val, test)
                    dst_folder = caminho_destino / info["destino"] / split_name
                    os.makedirs(dst_folder, exist_ok=True)
                    shutil.copy2(src_file, dst_folder / src_file.name)
                else:
                    print(f"Arquivo não encontrado: {src_file}")

    print(f"Arquivos de {ano} processados na nova estrutura!")

Arquivos de 2021 processados na nova estrutura!
Arquivos de 2022 processados na nova estrutura!
Arquivos de 2023 processados na nova estrutura!


In [ ]:
for split in ["train", "val", "test"]:
    pasta_segmentation = caminho_destino / split

    quantidade = sum(
        1
        for arquivo in pasta_segmentation.rglob("*.jpg")
    )

    print(f"{split}: {quantidade} imagens")

train: 0 imagens
val: 0 imagens
test: 0 imagens


In [ ]:
originais = sum(
    1 for arq in (caminho_dataset).rglob("*.jpg")
)

particionados = sum(
    1 for arq in (caminho_destino).rglob("*.jpg")
)

print("Originais:", originais)
print("Particionados:", particionados)
print("OK?" , originais == particionados)

Originais: 750
Particionados: 750
OK? True


In [ ]:
#salvar pasta no pc
shutil.make_archive(
    "Eliteserien_Particionado",
    "zip",
    "Eliteserien_Particionado"
)

'/home/bgds.eng22/Eliteserien_Particionado.zip'